# 교시별 실습 Notebook

d2-p06

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더를 확인했습니다.")

## 선행 셀 준비

d2-p04-05의 기본 셀입니다. 이 Notebook을 단독으로 열어도 필요한 변수를 준비할 수 있도록 포함했습니다.

In [ ]:
from pathlib import Path
import pandas as pd

titanic = pd.read_excel(Path("raw/기존강사 수업자료/titanic.xlsx"))
excluded = ["name", "ticket", "cabin", "boat", "body", "home.dest"]
model_data = titanic.drop(columns=excluded)
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = model_data[features].copy()
y = model_data["survived"].astype(int)
print("입력:", X.shape, "정답:", y.shape)
print(X.isna().sum())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ["pclass", "age", "sibsp", "parch", "fare"]
categorical = ["sex", "embarked"]

def make_preprocessor():
    return ColumnTransformer([
        ("numeric", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric),
        ("category", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
    ])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42,
)
print("훈련:", X_train.shape, "테스트:", X_test.shape)
print("훈련 생존율:", round(y_train.mean(), 4))
print("테스트 생존율:", round(y_test.mean(), 4))
assert set(X_train.index).isdisjoint(X_test.index)

In [ ]:
prepared = make_preprocessor()
train_array = prepared.fit_transform(X_train)
test_array = prepared.transform(X_test)
print("원래 훈련 입력:", X_train.shape)
print("변환 후 훈련 입력:", train_array.shape)
print("변환 후 테스트 입력:", test_array.shape)
print(prepared.get_feature_names_out())

# 요약

같은 Titanic 훈련·테스트 분리로 Logistic Regression, Decision Tree, RandomForest를 학습하고 다수 클래스 기준과 비교합니다.

## 2일차 6교시 — 세 분류 모델과 기준 모델 비교

2026년 10월 7일(수) · 2일차 6교시 · 1차 초안. 공식 안내문은 일차별 단원과 총시간을 정하며, 이 페이지의 교시별 묶음은 수업 진행을 위한 제안입니다. 같은 실습을 이어가는 교시는 한 페이지로 묶었습니다.

## 시작 전에

4~5교시의 `day2_titanic_ml` Notebook을 이어 씁니다. `pd`, `Pipeline`, `make_preprocessor`, `X_train`, `X_test`, `y_train`, `y_test`가 준비되어 있어야 합니다. 새 Notebook 또는 재시작한 Kernel에서는 4~5교시의 모든 코드 셀을 먼저 순서대로 실행합니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 수업 폴더 `kpc-lec`를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행 제안

- 세 모델의 판단 방식 비교 → Dummy 기준 세우기 → 같은 분리로 학습·평가 → 혼동행렬 읽기 → 트리 깊이에 따른 훈련/테스트 차이 확인.

## 5. Logistic Regression, Decision Tree, RandomForest 모델 학습

Logistic Regression은 선형 점수에서 생존 확률을 계산하고, Decision Tree는 조건을 나누며, RandomForest는 여러 트리의 결과를 결합합니다. 같은 데이터 분리와 같은 전처리 원칙으로 비교합니다. 먼저 다수 클래스를 항상 고르는 Dummy 모델을 기준으로 둡니다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

estimators = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=1),
}
fitted = {}
rows = []
for name, estimator in estimators.items():
    pipeline = Pipeline([("prepare", make_preprocessor()), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, prediction),
                 "F1_survived": f1_score(y_test, prediction, zero_division=0)})
    fitted[name] = pipeline
scores = pd.DataFrame(rows).set_index("model")
print(scores.round(4))
print("Logistic Regression 혼동행렬 (행=실제, 열=예측, 순서=0·1):")
print(confusion_matrix(y_test, fitted["Logistic Regression"].predict(X_test), labels=[0, 1]))

Accuracy는 전체에서 맞힌 비율이고 F1은 여기서 생존(1)의 precision과 recall을 함께 요약합니다. 기준 모델을 넘어서는지, 생존자를 놓치는 오류가 어떤지 함께 봅니다. 점수는 이 파일·분리·설정에서의 결과이며 새 데이터의 보장된 성능이 아닙니다.

**종합 실습:** Decision Tree의 `max_depth`를 2로 바꿔 같은 분리에서 다시 학습합니다. 원래 설정과 훈련 정확도·테스트 정확도를 표로 비교하고, 깊이를 늘릴수록 무조건 좋아진다고 말할 수 있는지 설명합니다.

<details>
<summary>종합 실습 예시 코드</summary>

```python
depth_rows = []
for depth in [2, 4, None]:
    model = Pipeline([("prepare", make_preprocessor()),
                      ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))])
    model.fit(X_train, y_train)
    depth_rows.append({"depth": str(depth), "train_accuracy": model.score(X_train, y_train),
                       "test_accuracy": model.score(X_test, y_test)})
print(pd.DataFrame(depth_rows).round(4))
```

</details>

## 확인 문제

1. 결측치 중앙값을 전체 데이터에서 먼저 계산하면 왜 문제가 될까요?
2. `boat`를 입력에 넣고 점수가 크게 오르면 좋은 모델인가요?
3. 다수 클래스를 고르는 모델과 비교하는 이유는 무엇인가요?

<details>
<summary>정답</summary>

1. 테스트 자료의 정보가 훈련 과정에 들어갑니다. `fit`은 훈련 자료에만 수행합니다.
2. 예측 당시 알 수 없는 사후 정보가 들어가면 실제 사용할 수 있는 예측 성능이 아닙니다.
3. 단순한 규칙보다 좋아졌는지 확인하고, 클래스 불균형으로 정확도가 높아 보이는 현상을 점검합니다.

</details>

## 직접 해보기와 정답

출력된 scores 표에서 기준 모델보다 Accuracy가 높고 생존 F1도 개선됐는지 확인합니다. 다음 셀로 실제 정답과 예측을 같은 행에 놓습니다.

In [ ]:
prediction = fitted["Logistic Regression"].predict(X_test)
example = X_test.copy()
example["actual_survived"] = y_test
example["predicted_survived"] = prediction
print(example.head(10))
print("틀린 예측 수:", (example["actual_survived"] != example["predicted_survived"]).sum())

질문: 훈련 점수는 1에 가까운데 테스트 점수는 낮다면 어떻게 설명하나요?

<details>
<summary>정답과 결과 해석 틀</summary>

훈련 자료의 작은 특징까지 외워 새로운 자료에 잘 적용하지 못하는 과적합 가능성이 있습니다. 깊이를 줄인 트리와 훈련·테스트 차이를 함께 비교합니다. “이 분리에서는 ○○ 모델의 Accuracy가 ○○, 생존 F1이 ○○였다. Dummy보다 ○○ 지표가 개선됐다. 새로운 모집단에서도 같은 성능이 유지되는지는 별도 검증이 필요하다.”처럼 직접 출력된 숫자를 사용합니다.

</details>

## 흔한 오류와 해결

| 증상 | 해결 |
| --- | --- |
| NameError: make_preprocessor | 4~5교시 전처리 함수 셀부터 실행합니다. |
| Kernel이 오래 실행 중 | 셀 왼쪽 실행 표시를 확인하고 완료를 기다립니다. 중복 실행하지 않습니다. |
| 모든 모델에서 점수가 비정상적으로 매우 높음 | 정답이나 boat·body 같은 사후 정보가 X에 들어갔는지 확인합니다. |
| 기준 모델 F1이 0 | 생존을 하나도 예측하지 않는 기준 규칙의 결과입니다. 실행 오류가 아닙니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 네 모델의 Accuracy·생존 F1 표를 만들었습니다.
- [ ] 혼동행렬의 행은 실제, 열은 예측임을 설명했습니다.
- [ ] 트리 깊이와 훈련/테스트 점수 차이를 비교했습니다.
- [ ] 점수가 새 자료의 성능을 보장하지 않는다는 문장을 썼습니다.

## 기준 자료

공식 안내문 3쪽의 「Titanic 데이터 전처리 및 머신러닝 모델링」 3시간 중 마지막 1교시를 기준으로 구성했습니다. 기존 단원 자료의 실습 코드를 교시별로 나눈 초안입니다. 실행 결과의 행 수는 제공 파일을 기준으로 하며, 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 7일 4~5교시 — 전처리와 훈련·테스트 분리](surl:14899) · [다음: 10월 7일 7~8교시 — 신용카드 부도 예측과 평가](surl:14901)
<!-- period-navigation:end -->